# Summer Analytics Capstone Part 2

### Import Libraries


In [655]:
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

### Load Datasets

In [656]:
trip = pd.read_csv("public_trip_data.csv")

pred_trip = pd.read_csv("private_trip_data.csv")

### Segregate Features

In [657]:
forbidden = ["Departure_CO2e", "Return_CO2e", "Hotel_CO2e", "Spend_CO2e", "TotalCO2e"]

In [658]:
target = "HighCarbon"

In [659]:
categorical = ["DepartureLocationCountry", "DepartureLocationCity", "ArrivalLocationCountry", "ArrivalLocationCity", "OutOfPolicy", "BusinessUnit",
               "Route"]

In [660]:
numeric = ["HotelNights", "NetCosts", "CostPerNight", "ShippingType"]

In [661]:
features = categorical + numeric

### Build the main dataset

In [662]:
df = trip.copy()
df = df.drop(columns=forbidden)

pred_df = pred_trip.copy()

### Feature Engineering

In [663]:
df["Route"] = df["DepartureLocationCity"].astype(str) + "__" + df["ArrivalLocationCity"].astype(str)
df["CostPerNight"] = df["NetCosts"] / df["HotelNights"].replace(0, 1)

pred_df["Route"] = pred_df["DepartureLocationCity"].astype(str) + "__" + pred_df["ArrivalLocationCity"].astype(str)
pred_df["CostPerNight"] = pred_df["NetCosts"] / pred_df["HotelNights"].replace(0, 1)

### Converting Objects to Category

In [664]:
for c in categorical:
  df[c] = df[c].astype(str).astype("category")
  pred_df[c] = pred_df[c].astype(str).astype("category")

### Modelling - HistGradientBoostingClassifier with Tuned Hyperparams

In [665]:
model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_depth=6, l2_regularization=0.5, early_stopping=True,
                                       validation_fraction=0.15, n_iter_no_change=20, random_state=42)

### Train-Test Split with StratifiedKFold Cross Validation

In [666]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [667]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)

print(f"CV ROC-AUC: {cv_scores.mean():.5f}  (+/- {cv_scores.std():.5f})  folds={np.round(cv_scores, 5)}")

CV ROC-AUC: 0.99933  (+/- 0.00007)  folds=[0.99939 0.99926 0.99924 0.99935 0.9994 ]


### Training the model and deriving evaluation metrics

In [668]:
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

In [669]:
metrics = {"roc_auc": roc_auc_score(y_test, proba),
        "f1": f1_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
    }
cm = confusion_matrix(y_test, pred)

for k, v in metrics.items():
        print(f"{k:10s}: {v:.5f}")
print("Confusion matrix:")
print(cm)

roc_auc   : 0.99939
f1        : 0.98726
precision : 0.98984
recall    : 0.98469
Confusion matrix:
[[9760   33]
 [  50 3215]]


In [670]:
perm = permutation_importance(
        model, X_test, y_test, n_repeats=5, random_state=42, scoring="roc_auc", n_jobs=-1
    )
importance = (
        pd.Series(perm.importances_mean, index=X_test.columns)
        .sort_values(ascending=False)
    )
print(importance.to_string())

Route                       0.323094
ShippingType                0.125146
HotelNights                 0.001943
ArrivalLocationCountry      0.001381
ArrivalLocationCity         0.000437
DepartureLocationCountry    0.000433
NetCosts                    0.000016
OutOfPolicy                 0.000007
CostPerNight                0.000006
DepartureLocationCity       0.000000
BusinessUnit               -0.000019


### Train model on complete dataset

In [671]:
model.fit(X, y)


HistGradientBoostingClassifier(early_stopping=True, l2_regularization=0.5,
                               learning_rate=0.06, max_depth=6, max_iter=300,
                               n_iter_no_change=20, random_state=42,
                               validation_fraction=0.15)

In [672]:
X = pred_df[features]
preds = model.predict(X)

submission = pd.DataFrame({"TripID": pred_df["TripID"], "HighCarbon": preds})